# CSP Pipeline

This notebook loads the Layer E candidate slots, models the scheduling problem as a **Constraint Satisfaction Problem** (one variable per satellite), reduces it with AC-3 and splits it into independent sub-problems. Each sub-problem is exported as a terminal subgraph for the shared annealing notebook.

The CSP formulation is intentionally different from the MWIS one (`02.A_mwis_pipeline.ipynb`): the two notebooks tackle the classical problem decomposition with two different approaches. The output artifacts, instead, follow the same format, so both producers can be compared downstream.

## 1. Configuration

All run parameters (input file, run name, QPU limits) are read from `results/_config.json` (or from the file in the `SWE2_CONFIG` environment variable). Set `run_name_csp` to select the CSP experiment and `csp_slot_csv` to select its input; the CSP graphics notebook uses the same settings. A copy of the configuration used is saved as `run_config_log.json` in `results/runs/csp/pipeline_results/<run_name_csp>`.

In [5]:
from __future__ import annotations

import copy
import gzip
import json
import os
import sys
import time
from collections import Counter
from itertools import combinations
from pathlib import Path
from typing import Any

import networkx as nx
import pandas as pd
pd.set_option("display.max_columns", None)


def find_project_root(start: Path | None = None) -> Path:
    """
    Find the repository root so paths work without further trouble.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing both `data/` and `results/`.")

# loading project root dir
PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"

# project modules live in `src/`
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from CSP import CSP, CSP_Variable, Timeslot, ac3, split_connected_components


# --------- loading configuration environment ----------
# load .json config file (same file used by the MWIS and QUBO notebooks)
CONFIG_PATH = Path(os.environ.get("SWE2_CONFIG", RESULTS_DIR / "_config.json"))
run_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

# input: candidate slots produced by the data-wrangling notebooks
CSP_SLOT_CSV = PROJECT_ROOT / run_config["csp_slot_csv"]

# CSP experiment name and output directory
RUN_NAME = run_config["run_name_csp"]
RUN_DIR = RESULTS_DIR / "runs" / "csp" / "pipeline_results" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# QPU physical constraints (the CSP does not decompose further yet: oversized sub-problems are only flagged)
MAX_QPU_TERMINAL_NODES = run_config["max_qpu_terminal_nodes"]
MAX_QPU_TERMINAL_COUPLERS = run_config["max_qpu_terminal_couplers"]

# logging environment configuration
(RUN_DIR / "run_config_log.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

print(f"Loaded configuration from: {CONFIG_PATH.resolve()}")
print(f"CSP experiment: {RUN_NAME}")
print(f"Input dataset: {CSP_SLOT_CSV.resolve()}")
print(f"Artifacts will be saved to: {RUN_DIR.resolve()}")

PHASE_PROFILE_PATH = RUN_DIR / f'csp_phase_profile_{time.strftime("%Y%m%d_%H%M%S")}.jsonl'
PIPELINE_STARTED = time.perf_counter()

def record_phase_timing(phase: str, started: float, **details: Any) -> float:
    """
    Helper function.
    Get how much time the specified phase takes to run, and store it in the log file.
    """
    elapsed = time.perf_counter() - started
    event = {"phase": phase, "elapsed_seconds": round(elapsed, 3), **details}
    with PHASE_PROFILE_PATH.open("a", encoding="utf-8") as profile_file:
        profile_file.write(json.dumps(event) + "\n")
        profile_file.flush()
    print(f"[CSP timing] {phase}: {elapsed:.2f}s", flush=True)
    return elapsed

Loaded configuration from: D:\University\PoliMi\courses\SWE2\SWE2-research-project\results\_config.json
CSP experiment: latest_csp_run
Input dataset: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\starlink_scheduling_slots_weighted.csv
Artifacts will be saved to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\results\runs\csp\pipeline_results\latest_csp_run


## 2. Load candidate slots

The data-wrangling notebooks already created weighted candidate contacts: each raw visibility window was discretised into fixed-length candidate slots, one every `discretization_step_seconds`. Every slot inherits the priority, urgency and weight of its own parent window.

The CSP works on a discrete time axis: the start of each slot is mapped to the integer **timeframe** `floor(start_seconds / step)`. Window starts are not aligned to the step grid (about half of them start 60 s off it), so the floor is used to make the mapping deterministic: two slots starting within the same step-long frame fall in the same timeframe.

In [7]:
raw_slots = pd.read_csv(CSP_SLOT_CSV)
raw_slots.tail(10)

,slot_id,parent_window_id,source_interval_id,satellite_name,satellite_id,norad_id,antenna_id,station_id,station_name,start_time,end_time,start_time_hours,end_time_hours,duration_seconds,duration_hours,parent_visibility_start,parent_visibility_end,parent_visibility_duration_seconds,candidate_index_in_parent,max_elevation_deg,source_epoch,antenna_type,station_capacity,reconfiguration_time_seconds,reconfiguration_time_hours,is_operationally_available,priority,urgency,priority_norm,duration_norm,urgency_norm,elevation_norm,weight,scenario_tag,priority_mode,urgency_mode,weight_scope,min_contact_duration_seconds,discretization_step_seconds
2546,SLOT_002547,VIS_004008,VIS_004008,STARLINK-1008,44714,44714,ANT_GATEWAY_0034,GATEWAY_0034,"Broadview, IL",2026-05-31T03:55:00+00:00,2026-05-31T04:00:00+00:00,3.916667,4.0,300,0.083333,2026-05-31T03:55:00+00:00,2026-05-31T04:00:00+00:00,300,1,16.919,2026-06-18T02:01:59Z,dish,1,180,0.05,True,2,1,0.25,0.625,0.0,0.187989,28.755,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
2547,SLOT_002548,VIS_004010,VIS_004010,STARLINK-1278,45408,45408,ANT_GATEWAY_0034,GATEWAY_0034,"Broadview, IL",2026-05-31T03:55:00+00:00,2026-05-31T04:00:00+00:00,3.916667,4.0,300,0.083333,2026-05-31T03:55:00+00:00,2026-05-31T04:00:00+00:00,300,1,58.209,2026-06-17T21:51:31Z,dish,1,180,0.05,True,3,1,0.50,0.625,0.0,0.646767,44.593,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
2548,SLOT_002549,VIS_004011,VIS_004011,STARLINK-1368,45540,45540,ANT_GATEWAY_0034,GATEWAY_0034,"Broadview, IL",2026-05-31T03:55:00+00:00,2026-05-31T04:00:00+00:00,3.916667,4.0,300,0.083333,2026-05-31T03:55:00+00:00,2026-05-31T04:00:00+00:00,300,1,54.149,2026-06-17T21:17:38Z,dish,1,180,0.05,True,5,1,1.00,0.625,0.0,0.601656,66.642,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
2549,SLOT_002550,VIS_003984,VIS_003984,STARLINK-1008,44714,44714,ANT_GATEWAY_0036,GATEWAY_0036,"Brooklyn Park, MN",2026-05-31T03:55:00+00:00,2026-05-31T04:00:00+00:00,3.916667,4.0,300,0.083333,2026-05-31T03:53:00+00:00,2026-05-31T04:00:00+00:00,420,2,34.616,2026-06-18T02:01:59Z,dish,1,180,0.05,True,3,1,0.50,0.875,0.0,0.384622,48.221,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
2550,SLOT_002551,VIS_003986,VIS_003986,STARLINK-1368,45540,45540,ANT_GATEWAY_0036,GATEWAY_0036,"Brooklyn Park, MN",2026-05-31T03:55:00+00:00,2026-05-31T04:00:00+00:00,3.916667,4.0,300,0.083333,2026-05-31T03:53:00+00:00,2026-05-31T04:00:00+00:00,420,2,53.890,2026-06-17T21:17:38Z,dish,1,180,0.05,True,5,1,1.00,0.875,0.0,0.598778,72.863,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
2551,SLOT_002552,VIS_004013,VIS_004013,STARLINK-1132,45044,45044,ANT_GATEWAY_0038,GATEWAY_0038,"Bulla Bulling, Australia",2026-05-31T03:55:00+00:00,2026-05-31T04:00:00+00:00,3.916667,4.0,300,0.083333,2026-05-31T03:55:00+00:00,2026-05-31T04:00:00+00:00,300,1,65.195,2026-06-18T07:15:40Z,dish,1,180,0.05,True,4,1,0.75,0.625,0.0,0.724389,56.619,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
2552,SLOT_002553,VIS_004014,VIS_004014,STARLINK-1043,44748,44748,ANT_GATEWAY_0043,GATEWAY_0043,"Calais, ME",2026-05-31T03:55:00+00:00,2026-05-31T04:00:00+00:00,3.916667,4.0,300,0.083333,2026-05-31T03:55:00+00:00,2026-05-31T04:00:00+00:00,300,1,19.010,2026-06-17T23:49:26Z,dish,1,180,0.05,True,3,1,0.50,0.625,0.0,0.211222,40.237,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
2553,SLOT_002554,VIS_004015,VIS_004015,STARLINK-1327,45555,45555,ANT_GATEWAY_0044,GATEWAY_0044,"Caldera, Chile",202

In [ ]:
REQUIRED_COLUMNS = [
    "slot_id", "parent_window_id", "satellite_id", "satellite_name", "antenna_id", "station_id",
    "start_time_hours", "end_time_hours", "duration_hours",
    "reconfiguration_time_hours", "priority", "urgency", "weight",
]

raw_slots = pd.read_csv(CSP_SLOT_CSV)
slots = raw_slots[REQUIRED_COLUMNS].copy()

# the discretisation step is the time unit of the CSP
disc_step_s = int(raw_slots["discretization_step_seconds"].iloc[0])

# integer seconds avoid floating-point noise; floor (not round) keeps the mapping deterministic
start_seconds = (slots["start_time_hours"] * 3600).round().astype(int)
slots["timeframe"] = start_seconds // disc_step_s

slots.to_csv(RUN_DIR / "candidate_slots.csv", index=False)

print({
    "candidate_slots": len(slots),
    "satellites": slots["satellite_name"].nunique(),
    "antennas": slots["antenna_id"].nunique(),
    "source_intervals": slots["parent_window_id"].nunique(),
    "discretization_step_seconds": disc_step_s,
    "slots_off_step_grid": int((start_seconds % disc_step_s != 0).sum()),
    "time_horizon_hours": (round(slots["start_time_hours"].min(), 3), round(slots["end_time_hours"].max(), 3)),
})

## 3. Translating to CSP

| Entity | CSP element |
|---|---|
| Satellite *S* | `CSP_Variable` (one per unique satellite) |
| Candidate slot at antenna *A*, starting at time *T* | `Timeslot(antenna_id=A, timeframe=floor(T / step))` |
| All candidate slots of satellite *S* | Domain of variable *S* |
| Weight of a candidate slot (from its parent window) | Weight of that domain value |
| Two satellites share a `Timeslot` | *Constraint* (Diff formulation: they cannot both be assigned it) |

Model assumptions, intentionally different from the MWIS formulation:
- each satellite receives **at most one** contact over the whole horizon;
- two satellites compete for an antenna only when their slots start in the **same timeframe**: partially overlapping slots in different timeframes and the antenna reconfiguration time are not modelled. Section 3.1 measures how many physical conflicts this leaves out.

In [ ]:
phase_started = time.perf_counter()

# look-up dictionaries to index variables and antennas
# (names are sorted, so the same dataset always yields the same ids)
satellites = sorted(slots["satellite_name"].unique())
antennas = sorted(slots["antenna_id"].unique())
sat_to_id = {s: i for i, s in enumerate(satellites)}
ant_to_id = {a: i for i, a in enumerate(antennas)}

# build the CSP: one variable per satellite, its domain is the slots it may use
csp = CSP(max_vars=len(satellites))
# (var_id, Timeslot) -> slot_id, to map CSP values back to the candidate slots
slot_of_value: dict[tuple[int, Timeslot], str] = {}
collapsed_slots = 0     # slots of one satellite falling on an already used (antenna, timeframe)

for sat_name, sat_rows in slots.groupby("satellite_name", sort=True):
    # the variable weight is only a default: every value carries the weight of its own slot
    var = CSP_Variable(id=sat_to_id[sat_name], name=sat_name, weight=float(sat_rows["weight"].max()))
    for row in sat_rows.sort_values("weight", ascending=False).itertuples(index=False):
        value = Timeslot(antenna_id=ant_to_id[row.antenna_id], timeframe=int(row.timeframe))
        if (var.id, value) in slot_of_value:
            collapsed_slots += 1    # keep the heaviest slot (rows are sorted by weight)
            continue
        var.add_value(value, weight=float(row.weight))
        slot_of_value[(var.id, value)] = row.slot_id
    # constraints are implicit: add_var links variables sharing a Timeslot (Diff formulation)
    csp.add_var(var)

record_phase_timing("csp_build", phase_started, variables=len(csp.vars), constraints=csp.num_constraints())


# --- prettier summary print ---
def print_summary(label, summ):
    print(f'\n{"-"*46}')
    print(f'  {label}')
    print(f'{"-"*46}')
    for k, v in summ.items():
        val = f'{v:.4f}' if isinstance(v, float) else str(int(v))
        print(f'  {k:<24} {val}')

summary_before = csp.compute_summary()
print_summary("CSP  —  BEFORE preprocessing", summary_before)
print(f"\nSlots collapsed on an existing (antenna, timeframe) of the same satellite: {collapsed_slots}")

# AC-3 prunes domains in place, so keep an untouched copy to compare against
csp_before = copy.deepcopy(csp)

### 3.1 Conflicts outside the CSP model

The Diff constraint only captures slots that start in the same timeframe on the same antenna. The cell below counts the pairs of slots of different satellites, on the same antenna, that physically conflict (they overlap, or leave less than the reconfiguration time between them) but are **not** linked by a CSP constraint. A CSP solution may therefore contain such pairs.

In [ ]:
phase_started = time.perf_counter()
physical_conflicts = 0
captured_conflicts = 0

for _, group in slots.groupby("antenna_id", sort=False):
    rows = list(group.sort_values("start_time_hours").itertuples(index=False))
    max_setup = group["reconfiguration_time_hours"].max()
    for i, first in enumerate(rows):
        for second in rows[i + 1:]:
            # sorted by start: once a slot starts after `first` ended plus the longest setup, all next ones do too
            if second.start_time_hours >= first.end_time_hours + max_setup - 1e-9:
                break
            # conflict: overlap, or a gap shorter than the setup time required by the pair
            setup = max(first.reconfiguration_time_hours, second.reconfiguration_time_hours)
            if second.start_time_hours >= first.end_time_hours + setup - 1e-9:
                continue
            if first.satellite_name == second.satellite_name:
                continue    # same satellite: already excluded by the at-most-one rule
            physical_conflicts += 1
            captured_conflicts += int(first.timeframe == second.timeframe)

missed_conflicts = physical_conflicts - captured_conflicts
record_phase_timing("model_gap_check", phase_started, missed_conflicts=missed_conflicts)
print({
    "physical_antenna_conflicts": physical_conflicts,
    "captured_by_diff_constraints": captured_conflicts,
    "not_captured": missed_conflicts,
})

## 4. Dimensionality Reduction

The problem's complexity reduction is composed of two phases.

### AC-3 Arc Consistency

AC-3 iteratively:
1. Removes unsupported values from domains (a timeslot is unsupported when a neighbour can *only* take that same timeslot)
2. Prunes constraints between variables whose domains have become disjoint
3. Propagates changes to neighbours

AC-3 reasons as if every satellite had to be served. On real data this is not always possible: some satellites can only use a single timeslot, which another satellite can also only use. AC-3 is run with `allow_wipeout=True`, so a satellite whose domain is wiped out is marked **unschedulable** and the pruning continues on the rest of the problem.

Since the downstream schedule may leave satellites out, this reduction is a heuristic: a value pruned because a neighbour "must" take it might still be used by the best schedule that skips that neighbour.

### Connected-Component Decomposition

After AC-3, variables with no remaining shared slots are completely independent. Splitting the CSP into connected components allows each sub-problem to be sent separately to the quantum annealer, reducing the QUBO size per call.

In [ ]:
# AC-3 enforces arc consistency: it removes from every domain the slots that
# cannot appear in a schedule serving every satellite.
phase_started = time.perf_counter()
all_satellites_servable = ac3(csp, allow_wipeout=True)
record_phase_timing("ac3", phase_started)

unschedulable = sorted(v.name for v in csp.vars if not v.domain)
print(f"Every satellite can be served : {all_satellites_servable}")
print(f"Unschedulable satellites      : {len(unschedulable)} {unschedulable}")

summary_after_ac3 = csp.compute_summary()
print_summary("CSP  —  AFTER AC-3", summary_after_ac3)

# how much did AC-3 actually reduce? (fewer domain values -> fewer QUBO variables downstream)
pruned = [
    (v.name, len(csp_before.get_var_by_id(v.id).domain), len(v.domain))
    for v in csp.vars
    if len(v.domain) < len(csp_before.get_var_by_id(v.id).domain)
]
print(f"\nSatellites with pruned domains : {len(pruned)}")
print(f"Domain values removed by AC-3  : {sum(b - a for _, b, a in pruned)}")
print(f"Constraints removed by AC-3    : {int(summary_before['num_const'] - summary_after_ac3['num_const'])}")
if pruned:
    print("\nPruned variables (name, before, after):")
    for name, b, a in pruned:
        print(f"  {name:<20}  {b} -> {a}  (-{b-a})")

In [ ]:
# satellites that share no timeslot can never conflict: the constraint graph splits
# into independent sub-problems, each solvable on its own and merged back at the end.
# Unschedulable satellites have an empty domain and are left out.
phase_started = time.perf_counter()
con_components = [
    component for component in split_connected_components(csp)
    if any(v.domain for v in component.vars)
]
# largest first, ties broken by the smallest variable id (deterministic ids)
sorted_components = sorted(con_components, key=lambda c: (-len(c.vars), min(v.id for v in c.vars)))
record_phase_timing("connected_components", phase_started, component_count=len(sorted_components))

sizes = [len(c.vars) for c in sorted_components]
print(f"Connected components after AC-3 : {len(sorted_components)}")
print(f"  Sizes (# satellites)  : {sizes}")
print(f"  Largest component     : {sizes[0]} satellites")
print(f"  Singleton components  : {sum(1 for s in sizes if s == 1)}")

In [ ]:
# ----------------------------------------------------------
# Export the CSP structure for the graphics notebook
# ----------------------------------------------------------

# one row per satellite: domain and degree, before and after AC-3
variables_rows = []
for before in csp_before.vars:
    after = csp.get_var_by_id(before.id)
    variables_rows.append({
        "var_id": before.id,
        "name": before.name,
        "weight": before.weight,
        "domain_size_before": len(before.domain),
        "domain_size_after": len(after.domain),
        "degree_before": csp_before.degree(before),
        "degree_after": csp.degree(after),
        "unschedulable": not after.domain,
    })
pd.DataFrame(variables_rows).to_csv(RUN_DIR / "csp_variables.csv", index=False)

# one row per constraint (satellite pair sharing a timeslot), for both stages
constraint_rows = [
    {"stage": stage, "u_id": v.id, "v_id": u.id}
    for stage, problem in (("before", csp_before), ("after", csp))
    for v in problem.vars
    for u in problem.neighbors(v)
    if u.id > v.id
]
pd.DataFrame(constraint_rows).to_csv(RUN_DIR / "csp_constraints.csv", index=False)

# one row per connected component found after AC-3 (largest first)
components_rows = [
    {
        "component_id": f"component_{index:03d}",
        "num_vars": len(component.vars),
        "num_constraints": component.num_constraints(),
        "qubo_vars": sum(len(v.domain) for v in component.vars),
    }
    for index, component in enumerate(sorted_components, start=1)
]
pd.DataFrame(components_rows).to_csv(RUN_DIR / "csp_components.csv", index=False)

print(f"Exported CSP structure to: {RUN_DIR.resolve()}")

## 5. Hand-over to the QUBO notebook

Each sub-problem is exported as a **conflict graph**, the format `3_qubo_annealing.ipynb` reads: one node per `(satellite, timeslot)` pair, identified by its candidate `slot_id` and weighted by that slot's weight, and one edge per pair that cannot both be selected.

| CSP element | Conflict graph |
|---|---|
| Binary variable `x[v, t]` | Node (`slot_id`) |
| Reward for scheduling satellite *v* at *t* | Node weight (weight of that slot) |
| At most one slot per satellite | Edge between the nodes of the same satellite (`same satellite`) |
| Diff constraint (one satellite per timeslot) | Edge between the nodes sharing a timeslot (`same timeslot`) |

An independent set of this graph is exactly a feasible assignment. As in the MWIS pipeline, isolated nodes (slots with no conflicts) are selected directly (`forced_selected`) and removed from the graph sent to the annealer.

The artifacts follow the MWIS pipeline format: `terminal_subgraphs.json`, `conflict_graph_nodes.csv` / `conflict_graph_edges.csv`, `components_summary.csv` and `reconstruction_metadata.json.gz`. Here every component is a single terminal: sub-problems larger than the QPU limits are flagged, not decomposed further.

In [ ]:
slot_attributes = slots.set_index("slot_id")


def component_to_conflict_graph(component: CSP) -> nx.Graph:
    """Turn one CSP component into a weighted conflict graph with one node per (satellite, timeslot)."""
    graph = nx.Graph()
    slot_users: dict[Timeslot, list[str]] = {}     # timeslot -> nodes competing for it

    for v in component.vars:
        var_nodes = []
        for t in sorted(v.domain, key=lambda t: (t.antenna_id, t.timeframe)):
            slot_id = slot_of_value[(v.id, t)]
            row = slot_attributes.loc[slot_id]
            graph.add_node(
                slot_id,
                source_interval_id=row.source_interval_id,
                satellite_id=str(row.satellite_id),
                satellite_name=v.name,
                antenna_id=row.antenna_id,
                station_id=row.station_id,
                start_time=float(row.start_time_hours),
                end_time=float(row.end_time_hours),
                duration=float(row.duration_hours),
                reconfiguration_time=float(row.reconfiguration_time_hours),
                priority=float(row.priority),
                urgency=float(row.urgency),
                weight=float(v.value_weight(t)),
                var_id=v.id,
                timeframe=t.timeframe,
            )
            var_nodes.append(slot_id)
            slot_users.setdefault(t, []).append(slot_id)

        # at most one slot per satellite
        for u, w in combinations(var_nodes, 2):
            graph.add_edge(u, w, reason="same satellite")

    # Diff constraint: a timeslot serves at most one satellite
    for users in slot_users.values():
        for u, w in combinations(users, 2):
            graph.add_edge(u, w, reason="same timeslot")

    return graph


def component_summary(component_id: str, graph: nx.Graph) -> dict[str, Any]:
    weights = [data["weight"] for _, data in graph.nodes(data=True)]
    return {
        "component_id": component_id,
        "node_count": graph.number_of_nodes(),
        "edge_count": graph.number_of_edges(),
        "density": nx.density(graph) if graph.number_of_nodes() > 1 else 0.0,
        "total_weight": sum(weights),
        "max_weight": max(weights, default=0.0),
        "satellite_count": len({data["satellite_id"] for _, data in graph.nodes(data=True)}),
        "antenna_count": len({data["antenna_id"] for _, data in graph.nodes(data=True)}),
    }


def graph_fits_qpu_limits(graph: nx.Graph) -> bool:
    """
    Return whether both terminal-size limits are satisfied.
    """
    return (
        graph.number_of_nodes() <= MAX_QPU_TERMINAL_NODES
        and graph.number_of_edges() <= MAX_QPU_TERMINAL_COUPLERS
    )


# Convert a graph to plain JSON: nodes and edges with their attributes
def serialize_graph(graph: nx.Graph) -> dict[str, Any]:
    return {
        "nodes": [{"id": node, **data} for node, data in graph.nodes(data=True)],
        "edges": [{"source": u, "target": v, **data} for u, v, data in graph.edges(data=True)],
    }

In [ ]:
phase_started = time.perf_counter()
components = []             # one record per component: id, full conflict graph, summary
terminal_subgraphs = []     # non-empty terminals, read by the annealing notebook
metadata_nodes = []         # one terminal node per component, read by reconstruction tools

for index, component in enumerate(sorted_components, start=1):
    component_id = f"component_{index:03d}"
    graph = component_to_conflict_graph(component)
    components.append({"component_id": component_id, "graph": graph, "summary": component_summary(component_id, graph)})

    # isolated slots have no conflict: select them directly, as the MWIS preprocessing does
    forced_selected = sorted(nx.isolates(graph))
    forced_objective = sum(graph.nodes[node]["weight"] for node in forced_selected)
    reduced = graph.subgraph(set(graph.nodes) - set(forced_selected))
    terminal_reason = (
        "resolved by exact preprocessing" if reduced.number_of_nodes() == 0
        else "fits QPU limits" if graph_fits_qpu_limits(reduced)
        else "exceeds QPU limits (no further CSP decomposition)"
    )
    terminal_id = f"{component_id}_dnode_0001"

    if reduced.number_of_nodes() > 0:
        terminal_subgraphs.append({
            "terminal_id": terminal_id,
            "original_component_id": component_id,
            "depth": 0,
            "node_count": reduced.number_of_nodes(),
            "edge_count": reduced.number_of_edges(),
            "forced_objective": forced_objective,
            "forced_selected": forced_selected,
            "terminal_reason": terminal_reason,
            "graph": serialize_graph(reduced),
        })

    metadata_nodes.append({
        "node_id": terminal_id,
        "component_id": component_id,
        "node_type": "terminal",
        "depth": 0,
        "input_node_count": graph.number_of_nodes(),
        "input_edge_count": graph.number_of_edges(),
        "qpu_node_count": reduced.number_of_nodes(),
        "qpu_edge_count": reduced.number_of_edges(),
        "forced_selected": forced_selected,
        "forced_objective": float(forced_objective),
        "terminal_reason": terminal_reason,
        "terminal_id": terminal_id if reduced.number_of_nodes() else None,
    })

# full conflict graph (union of the components) for inspection and downstream notebooks
conflict_graph = nx.union_all([component["graph"] for component in components])
node_rows = [{"slot_id": node_id, **data} for node_id, data in conflict_graph.nodes(data=True)]
edge_rows = [{"source": u, "target": v, **data} for u, v, data in conflict_graph.edges(data=True)]
pd.DataFrame(node_rows).to_csv(RUN_DIR / "conflict_graph_nodes.csv", index=False)
pd.DataFrame(edge_rows).to_csv(RUN_DIR / "conflict_graph_edges.csv", index=False)

components_summary = pd.DataFrame([component["summary"] for component in components])
components_summary.to_csv(RUN_DIR / "components_summary.csv", index=False)

(RUN_DIR / "terminal_subgraphs.json").write_text(json.dumps(terminal_subgraphs, indent=2), encoding="utf-8")

# Reconstruction map: one terminal root per component (no separator decomposition in the CSP pipeline)
reconstruction_metadata = {
    "schema_version": 1,
    "producer": "csp_pipeline",
    "run_name": RUN_NAME,
    "objective": "maximum-weight assignment, at most one timeslot per satellite",
    "terminal_subgraphs_file": "terminal_subgraphs.json",
    "components": [
        {"component_id": node["component_id"], "root_node_id": node["node_id"], "status": "complete"}
        for node in metadata_nodes
    ],
    "nodes": metadata_nodes,
}
with gzip.open(RUN_DIR / "reconstruction_metadata.json.gz", "wt", encoding="utf-8", compresslevel=6) as metadata_file:
    json.dump(reconstruction_metadata, metadata_file)

oversized = [t for t in terminal_subgraphs if t["terminal_reason"].startswith("exceeds")]
record_phase_timing("conflict_graph_and_terminal_export", phase_started, terminal_subgraphs=len(terminal_subgraphs))

print({
    "nodes": conflict_graph.number_of_nodes(),
    "edges_couplers": conflict_graph.number_of_edges(),
    "edge_reasons": dict(Counter(data["reason"] for _, _, data in conflict_graph.edges(data=True))),
    "components": len(components),
    "terminal_subgraphs": len(terminal_subgraphs),
    "terminals_exceeding_qpu_limits": len(oversized),
    "largest_terminal_nodes": max((t["node_count"] for t in terminal_subgraphs), default=0),
    "largest_terminal_couplers": max((t["edge_count"] for t in terminal_subgraphs), default=0),
    "qpu_limits": (MAX_QPU_TERMINAL_NODES, MAX_QPU_TERMINAL_COUPLERS),
})
components_summary.head(10)

## 6. Final summary table

This table is the main compact output of the notebook. The graphics and QUBO notebooks read the artifacts saved above instead of rebuilding this pipeline.

In [ ]:
domain_values_before = sum(len(v.domain) for v in csp_before.vars)
domain_values_after = sum(len(v.domain) for v in csp.vars)

# Final metrics as (name, value, meaning)
final_summary_rows = [
    ("Candidate slots", len(slots), "Rows of the input dataset."),
    ("Satellites (CSP variables)", len(csp.vars), "One variable per satellite."),
    ("Unschedulable satellites", len(unschedulable), "Satellites whose domain was wiped out by AC-3."),
    ("Domain values before AC-3", domain_values_before, "(satellite, timeslot) pairs before preprocessing."),
    ("Domain values after AC-3", domain_values_after, "(satellite, timeslot) pairs left after AC-3, i.e. QUBO variables."),
    ("Constraints before AC-3", int(summary_before["num_const"]), "Satellite pairs sharing at least one timeslot."),
    ("Constraints after AC-3", int(summary_after_ac3["num_const"]), "Satellite pairs still sharing a timeslot after AC-3."),
    ("Physical antenna conflicts not captured", missed_conflicts, "Overlap/reconfiguration conflicts between different satellites outside the Diff constraint."),
    ("Number of components (total)", len(components), "Connected components of the constraint graph after AC-3."),
    ("Number of couplers (total)", conflict_graph.number_of_edges(), "Conflict edges in the full graph."),
    ("Terminal subgraphs", len(terminal_subgraphs), "Non-empty sub-problems exported for annealing."),
    ("Terminal subgraphs exceeding QPU limits", len(oversized), "Sub-problems that would need a further decomposition."),
    ("Forced-selected objective", sum(node["forced_objective"] for node in metadata_nodes), "Weight of isolated slots selected without annealing."),
]

final_summary = pd.DataFrame(final_summary_rows, columns=["metric", "value", "meaning"])
phase_started = time.perf_counter()
final_summary.to_csv(RUN_DIR / "final_summary.csv", index=False)
record_phase_timing("final_reporting_export", phase_started, metrics=len(final_summary))
record_phase_timing("pipeline_total_so_far", PIPELINE_STARTED, run_name=RUN_NAME)

# Report the shared annealing target without changing it (it may intentionally target MWIS).
print(f"CSP experiment saved: {RUN_NAME}")
annealing_target = run_config.get("qubo_input_run")
if annealing_target and (RESULTS_DIR / annealing_target).resolve() == RUN_DIR.resolve():
    print("The shared annealing input matches this CSP experiment.")
else:
    print(f"The shared annealing notebook currently targets: {annealing_target or '(not configured)'}")
    print("To anneal this CSP experiment, set qubo_input_run to:")
    print(RUN_DIR.relative_to(RESULTS_DIR).as_posix())

final_summary